# DNT Organoid AI — High-Memory Cloud Quantification Bridge
## Reference Index Preparation & Persistent Storage Synchronization

**Study Accession:** BioProject `PRJNA802072` (SRA `SRP357550`)
**Target Runtime:** Google Colab High-RAM Instance (83.5 GB RAM, 12 vCPUs, ~235 GB Local NVMe/SSD)
**Storage Architecture:** Local NVMe/SSD for High-IOPS Active Processing + Google Drive for Persistent Verification Checkpoints
**Reference Build:** Ensembl GRCh38 Release 112 (Primary Assembly + Annotation GTF + Deterministic BED12)

> **GOVERNANCE MANDATE:**
> 1. **Local Scratch for Active Work:** Active STAR indexing, FASTQ streaming, and samtools sorting execute on local NVMe (`/content`) to avoid Google Drive FUSE latency and rate limits.
> 2. **Integrity-Verified Persistence:** All persistent transfers to Google Drive are verified via end-to-end SHA-256 comparison before local cleanup.
> 3. **Capacity Pre-Flight:** Google Drive storage is programmatically probed before transfers to ensure sufficient free space + safety margin.
> 4. **Scientific Review Gate:** Processing strictly pauses after reference-index generation and validation. No FASTQ sequencing libraries are downloaded until review is complete.

### Stage 1: Optional Google Drive Mount & Persistence Configuration
Mount your Google Drive to store verified checkpoints and indexes across sessions. If you are running locally or choose not to use Google Drive, the pipeline falls back seamlessly to local persistent storage.

In [ ]:
import os
import sys
from pathlib import Path
import shutil

# Check if running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("Requesting Google Drive mount authorization...")
    drive.mount('/content/drive')
    # Configurable persistent root (never hardcoded to an individual user)
    PERSISTENCE_ROOT = Path('/content/drive/MyDrive/DNT_Organoid_AI_Persistence')
    print(f"[OK] Google Drive mounted. Persistent storage root: {PERSISTENCE_ROOT}")
else:
    PERSISTENCE_ROOT = Path('/tmp/DNT_Organoid_AI_Persistence')
    print(f"Running outside Colab. Fallback persistent storage root: {PERSISTENCE_ROOT}")

PERSISTENCE_ROOT.mkdir(parents=True, exist_ok=True)

# Check available capacity on persistent storage
total, used, free = shutil.disk_usage(PERSISTENCE_ROOT)
print(f"Persistent Storage Status: {round(free / (1024**3), 2)} GB free of {round(total / (1024**3), 2)} GB total")

### Stage 2: Fast Bioinformatics Toolchain Provisioning (Micromamba)
Install the pinned bioinformatics toolchain into Colab runtime:
- **STAR 2.7.11b** (Splice-aware reference aligner)
- **fastp 0.23.4** (Adapter and poly-G trimmer)
- **subread 2.0.6 / featureCounts** (Gene-level summarization)
- **samtools 1.19.2** (BAM sorting and indexing)
- **rseqc 5.0.2 / infer_experiment.py** (Strandedness verification)

In [ ]:
%%bash
# Fast micromamba provisioning in /content/micromamba
if [ ! -f "/content/micromamba/bin/micromamba" ]; then
  echo "Downloading standalone micromamba binary..."
  mkdir -p /content/micromamba
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/micromamba bin/micromamba
fi

if [ ! -d "/content/dnt_env" ]; then
  echo "Creating pinned bioinformatics environment (/content/dnt_env)..."
  /content/micromamba/bin/micromamba create -y -p /content/dnt_env \
    -c conda-forge -c bioconda \
    star=2.7.11b \
    fastp=0.23.4 \
    subread=2.0.6 \
    samtools=1.19.2 \
    rseqc=5.0.2 \
    python=3.12 \
    pyyaml \
    pytest
fi

# Test tools in environment
export PATH="/content/dnt_env/bin:$PATH"
echo "Toolchain Verification:"
STAR --version
fastp --version
featureCounts -v 2>&1 | head -n 2
samtools --version | head -n 1
infer_experiment.py --version 2>&1

### Stage 3: Repository Setup on Local High-Speed SSD
Clone or link the repository to `/content/DNT-Organoid-AI`. All active files, scratch calculations, and intermediate indexing buffers reside here on local SSD.

In [ ]:
import os
import sys
from pathlib import Path

# Prepend the conda environment to system PATH
os.environ["PATH"] = f"/content/dnt_env/bin:{os.environ.get('PATH', '')}"

REPO_DIR = Path("/content/DNT-Organoid-AI")
if not REPO_DIR.exists():
    print(f"Cloning DNT-Organoid-AI repository to local SSD ({REPO_DIR})...")
    !git clone -b feature/fastq-sra-real-data-validation https://github.com/bioprintingLabsInc/DNT-Organoid-AI.git /content/DNT-Organoid-AI
else:
    print(f"Repository already present on local disk: {REPO_DIR}")

# Add repo root to python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%cd /content/DNT-Organoid-AI
!git status -s

### Stage 4: Hardware Probing & Execution Profile Preflight
Probe the runtime hardware baseline and verify compliance with the `colab_highmem` (or `institutional_hpc`) profile.

In [ ]:
from src.quantification_bridge.pipeline import QuantificationBridgePipeline
from src.quantification_bridge.profiles import get_execution_profile, resolve_pipeline_paths

# Load colab_highmem execution profile
pipeline = QuantificationBridgePipeline(repo_root=REPO_DIR, profile_name="colab_highmem")
env_metrics = pipeline.record_environment_metrics()

print("=" * 80)
print("HARDWARE BASELINE AUDIT:")
print(f"OS / Kernel:           {env_metrics['uname']}")
print(f"Physical RAM:          {env_metrics['physical_ram_gb']} GB (Requirement: >= 75.0 GB)")
print(f"CPU Cores:             {env_metrics['cpu_cores']} (Requirement: >= 8)")
print(f"Local Scratch Free:    {env_metrics['scratch_disk_free_gb']} GB (Requirement: >= 150.0 GB)")
print("=" * 80)

# Run formal profile preflight checks
preflight_res = pipeline.run_profile_checks(enforce_physical_ram=True)
print(f"Preflight Compatibility: {preflight_res.is_compatible}")
print(f"Fatal Errors:            {preflight_res.has_fatal_errors}")
for finding in preflight_res.findings:
    print(f"  [{finding.severity}] {finding.code}: {finding.message}")

assert not preflight_res.has_fatal_errors, "Preflight checks failed! Cannot proceed."

### Stage 5: Reference Acquisition & Integrity Verification
Acquire Ensembl Release 112 GRCh38 primary assembly FASTA and GTF. Verify SHA-256 cryptographic checksums against `reference_manifest.json` before decompression.

In [ ]:
from src.quantification_bridge.checkpoints import CheckpointManager

paths = resolve_pipeline_paths(REPO_DIR)
checkpoint_mgr = CheckpointManager(
    local_checkpoint_dir=paths["data_dir"] / "checkpoints",
    persistent_checkpoint_dir=PERSISTENCE_ROOT / "checkpoints",
)

print("Acquiring and cryptographically verifying reference files...")
ref_res = pipeline.acquire_and_verify_references()
print("[OK] Reference Verification Complete:")
print(f"  FASTA SHA-256: {ref_res['fasta_sha256']} (VERIFIED)")
print(f"  GTF SHA-256:   {ref_res['gtf_sha256']} (VERIFIED)")
print(f"  BED12 SHA-256: {ref_res['bed12_sha256']} (VERIFIED)")

checkpoint_mgr.record_stage_completed("REFERENCES_VERIFIED", artifacts=[ref_res])

### Stage 6: Monitored STAR Reference-Index Generation (Local High-Speed SSD)
Generate the governed STAR genome index on the fast local NVMe/SSD using `colab_highmem` parameters:
- **RAM Limit:** 60,000,000,000 bytes (60 GB)
- **Threads:** 12
- **Splice Junction Overhang:** 149
- **sparseD:** Configurable (`1` for canonical dense reference standard, or `2`/`4` for memory-bounded evaluation)

In [ ]:
import time

# Check if verified index already exists in persistent storage to save time
PERSISTENT_INDEX_DIR = PERSISTENCE_ROOT / "references" / "star_index"
SKIP_IF_ALREADY_ON_DRIVE = True

need_to_build = True
if SKIP_IF_ALREADY_ON_DRIVE and (PERSISTENT_INDEX_DIR / "SA").is_file():
    print(f"[FOUND] Existing STAR index detected on persistent storage: {PERSISTENT_INDEX_DIR}")
    print("Restoring index to fast local SSD (/content)... (approx 1-3 minutes)")
    from src.quantification_bridge.checkpoints import sync_and_verify_directory
    res_sync = sync_and_verify_directory(PERSISTENT_INDEX_DIR, paths["star_index_dir"], safety_margin_gb=5.0)
    print(f"[RESTORED] Restored {res_sync['total_files']} files ({round(res_sync['total_bytes'] / (1024**3), 2)} GB) in {res_sync['total_seconds']}s.")
    need_to_build = False

if need_to_build:
    print("Starting fresh STAR genomeGenerate on local SSD...")
    t0 = time.time()
    idx_res = pipeline.build_star_index(
        threads=12,
        sparse_d=1,  # Canonical dense reference standard on High-RAM Colab
        genome_sa_index_nbases=14,
        limit_ram=60000000000,
    )
    duration_min = (time.time() - t0) / 60
    print(f"[OK] STAR Index generation finished cleanly in {duration_min:.2f} minutes!")

### Stage 7: Index Completeness Audit & In-Memory Dry-Run Load Test
Verify that all required STAR binary index files exist, are non-empty, and load into memory without launch of full sequencing alignment.

In [ ]:
from scripts.run_colab_reference_indexing import verify_index_artifacts, lightweight_index_load_test

print("Verifying index artifact inventory...")
eval_res = verify_index_artifacts(paths["star_index_dir"])
assert eval_res["is_complete"], f"Index is incomplete! Missing: {eval_res['missing_files']}"

print(f"[OK] All {len(eval_res['file_details'])} required files verified present and valid:")
for fname, detail in eval_res['file_details'].items():
    print(f"  - {fname:<25} {detail['size_mb']:>9.2f} MB")

print(f"\nTotal Local Index Size: {eval_res['total_index_gb']} GB")

# Lightweight dry-run load test
load_test = lightweight_index_load_test(paths["star_index_dir"], star_bin="STAR")
print(f"In-Memory Load Test Status: {load_test['load_test_passed']} (Exit Code: {load_test.get('exit_code')}, Time: {load_test['load_time_seconds']}s)")
assert load_test["load_test_passed"], "In-memory index load test failed!"

checkpoint_mgr.record_stage_completed(
    "STAR_INDEX_BUILT_AND_VERIFIED",
    artifacts=[{"dir": str(paths["star_index_dir"]), "total_gb": eval_res["total_index_gb"]}],
)

### Stage 8: Persistent Storage Synchronization (Capacity Check & SHA-256 Verification)
Before transferring the large index (~27.4 GB dense or ~10 GB sparse), programmatically probe Google Drive capacity. Once capacity is verified, copy every file and verify cryptographic SHA-256 integrity on the remote storage.

In [ ]:
from src.quantification_bridge.checkpoints import check_storage_capacity, sync_and_verify_directory

TARGET_PERSISTENT_INDEX = PERSISTENCE_ROOT / "references" / "star_index"
SAFETY_MARGIN_GB = 10.0

print("Checking persistent storage capacity before transfer...")
cap = check_storage_capacity(
    target_path=TARGET_PERSISTENT_INDEX,
    required_bytes=eval_res["total_index_bytes"],
    safety_margin_gb=SAFETY_MARGIN_GB,
)
print(f"Persistent Free Space:  {cap['free_gb']} GB")
print(f"Required Index Space:  {cap['required_gb']} GB (+ {SAFETY_MARGIN_GB} GB margin = {cap['total_needed_gb']} GB needed)")

print(f"\nSynchronizing index to {TARGET_PERSISTENT_INDEX} with end-to-end SHA-256 verification...")
sync_res = sync_and_verify_directory(
    src_dir=paths["star_index_dir"],
    dst_dir=TARGET_PERSISTENT_INDEX,
    safety_margin_gb=SAFETY_MARGIN_GB,
    delete_source_after=False,  # Retain local copy for active alignment
)

print("=" * 80)
print(f"[PERSISTENCE VERIFIED] Successfully synced {sync_res['total_files']} files to persistent store in {sync_res['total_seconds']}s.")
print("=" * 80)

checkpoint_mgr.record_stage_completed(
    "STAR_INDEX_PERSISTED_TO_GDRIVE",
    artifacts=[{"dir": str(TARGET_PERSISTENT_INDEX), "files": sync_res["total_files"]}],
)

## MANDATORY SCIENTIFIC REVIEW GATE
### EXECUTION HALT

In strict compliance with user governance:
- **Reference Indexing is Complete and Validated.**
- **Persistent Backup is Cryptographically Verified.**
- **DO NOT proceed to FASTQ sequencing library acquisition or alignment until this report is formally reviewed and approved.**
- Run the final cell below to produce the formal readiness decision table.

In [ ]:
print("=" * 80)
print("DNT ORGANOID AI - REFERENCE INDEX VERIFICATION SUMMARY")
print("=" * 80)
print(f"Repository Git Commit:        {os.popen('git rev-parse --short HEAD').read().strip()}")
print(f"Active Profile:               colab_highmem")
print(f"Reference Genome:             Ensembl GRCh38 release 112 (Homo sapiens)")
print(f"Total Local Index Size:       {eval_res['total_index_gb']} GB")
print(f"Total Persistent Index Size:  {eval_res['total_index_gb']} GB (Verified Match)")
print(f"Files Verified:               {len(eval_res['file_details'])}")
print(f"In-Memory Load Status:        PASSED")
print(f"Google Drive Checkpoint:      COMPLETED & PERSISTED")
print(f"Status:                       AWAITING_OWNER_PILOT_FASTQ_AUTHORIZATION")
print("=" * 80)
print("PIPELINE PAUSED: AWAITING USER APPROVAL TO BEGIN 2-SAMPLE TECHNICAL PILOT FASTQ DOWNLOAD.")